In [ ]:

from pathlib import Path
import pandas as pd

# Locate the original Excel dataset
file_path = Path("../data/raw/online_retail_II.xlsx")

print("File exists:", file_path.exists())
print("Dataset path:", file_path.resolve())
print("Pandas version:", pd.__version__)


In [ ]:

excel_file = pd.ExcelFile(file_path)
print(excel_file.sheet_names)


In [ ]:

sample = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010",
    nrows=10
)

print("Dataset loaded successfully!")
print("Shape:", sample.shape)
display(sample)


In [ ]:

print("Column names:")
print(sample.columns.tolist())

print("\nData types:")
print(sample.dtypes)

print("\nMissing values in these 10 rows:")
print(sample.isna().sum())

print("\nFirst 10 records:")
display(sample)


In [ ]:

from openpyxl import load_workbook

workbook = load_workbook(
    file_path,
    read_only=True,
    data_only=True
)

for sheet in workbook.worksheets:
    print(sheet.title, "Rows including header:", sheet.max_row)
    print(sheet.title, "Columns:", sheet.max_column)

workbook.close()



In [ ]:

import pandas as pd

file_path = Path("../data/raw/online_retail_II.xlsx")

# Read the first sheet in manageable chunks
chunk_reader = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010",
    chunksize=None
)


In [ ]:

from openpyxl import load_workbook
from collections import Counter

workbook = load_workbook(
    file_path,
    read_only=True,
    data_only=True
)

sheet = workbook["Year 2009-2010"]

headers = next(sheet.iter_rows(values_only=True))
missing_counts = Counter()
seen_rows = set()
duplicate_count = 0
row_count = 0

for row in sheet.iter_rows(min_row=2, values_only=True):
    row_count += 1

    for column, value in zip(headers, row):
        if value is None:
            missing_counts[column] += 1

    row_key = tuple(row)

    if row_key in seen_rows:
        duplicate_count += 1
    else:
        seen_rows.add(row_key)

print("Rows checked:", row_count)
print("\nMissing values:")
for column in headers:
    print(f"{column}: {missing_counts[column]}")

print("\nExact duplicate rows:", duplicate_count)

workbook.close()


In [ ]:

workbook = load_workbook(
    file_path,
    read_only=True,
    data_only=True
)

sheet = workbook["Year 2010-2011"]
headers = next(sheet.iter_rows(values_only=True))

missing_counts = Counter()
seen_rows = set()
duplicate_count = 0
row_count = 0

for row in sheet.iter_rows(min_row=2, values_only=True):
    row_count += 1

    for column, value in zip(headers, row):
        if value is None:
            missing_counts[column] += 1

    row_key = tuple(row)

    if row_key in seen_rows:
        duplicate_count += 1
    else:
        seen_rows.add(row_key)

print("Rows checked:", row_count)
print("\nMissing values:")
for column in headers:
    print(f"{column}: {missing_counts[column]}")

print("\nExact duplicate rows:", duplicate_count)

workbook.close()


In [ ]:
from pathlib import Path

processed_folder = Path("../data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)

print("Output folder ready:", processed_folder.resolve())

In [ ]:

import csv
from openpyxl import load_workbook

file_path = Path("../data/raw/online_retail_II.xlsx")
output_file = processed_folder / "retail_combined_raw.csv"

workbook = load_workbook(
    file_path,
    read_only=True,
    data_only=True
)

sheet_names = ["Year 2009-2010", "Year 2010-2011"]
total_rows = 0

with open(output_file, "w", newline="", encoding="utf-8-sig") as csvfile:
    writer = csv.writer(csvfile)
    headers_written = False

    for sheet_name in sheet_names:
        sheet = workbook[sheet_name]
        rows = sheet.iter_rows(values_only=True)
        headers = next(rows)

        if not headers_written:
            writer.writerow(headers)
            headers_written = True

        for row in rows:
            writer.writerow(row)
            total_rows += 1

        print(f"Finished: {sheet_name}")

workbook.close()

print("Total rows combined:", total_rows)
print("Saved to:", output_file.resolve())


In [ ]:

import sqlite3
import csv

input_file = processed_folder / "retail_combined_raw.csv"
cleaned_file = processed_folder / "retail_cleaned.csv"
db_file = processed_folder / "retail_cleaning_temp.db"

conn = sqlite3.connect(db_file)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS transactions (
    row_data TEXT PRIMARY KEY
)
""")

rows_read = 0
duplicates = 0

with open(input_file, "r", newline="", encoding="utf-8-sig") as f:
    reader = csv.reader(f)
    headers = next(reader)

    for row in reader:
        rows_read += 1
        row_text = __import__("json").dumps(row, ensure_ascii=False)

        cursor.execute(
            "INSERT OR IGNORE INTO transactions (row_data) VALUES (?)",
            (row_text,)
        )

        if cursor.rowcount == 0:
            duplicates += 1

        if rows_read % 10000 == 0:
            conn.commit()

conn.commit()

print("Rows checked:", rows_read)
print("Exact duplicate rows found:", duplicates)
print("Unique rows remaining:", rows_read - duplicates)


In [ ]:

import json

cursor.execute("""
SELECT COUNT(*) FROM transactions
""")
print("Unique records in database:", cursor.fetchone()[0])

with open(cleaned_file, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(headers)

    for (row_text,) in cursor.execute("SELECT row_data FROM transactions"):
        writer.writerow(json.loads(row_text))

print("Saved cleaned file:", cleaned_file.resolve())


In [ ]:

import pandas as pd

cleaned_file = processed_folder / "retail_cleaned.csv"

missing_counts = None
total_rows = 0

for chunk in pd.read_csv(cleaned_file, chunksize=50000):
    total_rows += len(chunk)
    counts = chunk.isna().sum()

    if missing_counts is None:
        missing_counts = counts
    else:
        missing_counts += counts

print("Total rows:", total_rows)
print("\nMissing values by column:")
print(missing_counts)



In [ ]:

negative_qty = 0
zero_price = 0
negative_price = 0
cancelled_invoices = 0

for chunk in pd.read_csv(
    cleaned_file,
    chunksize=50000,
    dtype={"Invoice": "string", "StockCode": "string"}
):
    negative_qty += (chunk["Quantity"] < 0).sum()
    zero_price += (chunk["Price"] == 0).sum()
    negative_price += (chunk["Price"] < 0).sum()
    cancelled_invoices += chunk["Invoice"].str.startswith(
        "C", na=False
    ).sum()

print("Negative quantities:", negative_qty)
print("Zero prices:", zero_price)
print("Negative prices:", negative_price)
print("Cancelled invoice rows:", cancelled_invoices)


In [ ]:

negative_price_rows = []

for chunk in pd.read_csv(
    cleaned_file,
    chunksize=50000,
    dtype={"Invoice": "string", "StockCode": "string"}
):
    rows = chunk[chunk["Price"] < 0]
    if not rows.empty:
        negative_price_rows.append(rows)

negative_price_df = pd.concat(
    negative_price_rows,
    ignore_index=True
)

display(negative_price_df)


In [ ]:

bad_debt_count = 0

for chunk in pd.read_csv(cleaned_file, chunksize=50000):
    bad_debt_count += (
        chunk["Description"].astype("string")
        .str.strip()
        .str.casefold()
        .eq("adjust bad debt")
        .sum()
    )

print("Bad-debt adjustment rows:", bad_debt_count)


In [ ]:

sales_file = processed_folder / "retail_sales_analysis.csv"

first_chunk = True
rows_saved = 0

for chunk in pd.read_csv(
    cleaned_file,
    chunksize=50000,
    dtype={"Invoice": "string", "StockCode": "string"}
):
    description = (
        chunk["Description"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    # Exclude accounting adjustments and cancelled invoices
    keep = (
        ~description.eq("adjust bad debt").fillna(False)
        & ~chunk["Invoice"].str.startswith("C", na=False)
    )

    sales_chunk = chunk.loc[keep].copy()

    sales_chunk.to_csv(
        sales_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
        encoding="utf-8-sig" if first_chunk else "utf-8"
    )

    rows_saved += len(sales_chunk)
    first_chunk = False

print("Sales-analysis rows:", rows_saved)
print("Saved to:", sales_file.resolve())


In [ ]:
sales_df = pd.read_csv(sales_file, nrows=10)

print("Shape of sample:", sales_df.shape)
print("Columns:", sales_df.columns.tolist())
display(sales_df.head())

In [ ]:
sales_df = pd.read_csv(
    sales_file,
    dtype={"Invoice": "string", "StockCode": "string"}
)

sales_df["InvoiceDate"] = pd.to_datetime(
    sales_df["InvoiceDate"],
    errors="coerce"
)

sales_df["Customer ID"] = pd.to_numeric(
    sales_df["Customer ID"],
    errors="coerce"
)

sales_df["Quantity"] = pd.to_numeric(
    sales_df["Quantity"],
    errors="coerce"
)

sales_df["Price"] = pd.to_numeric(
    sales_df["Price"],
    errors="coerce"
)

sales_df["Revenue"] = sales_df["Quantity"] * sales_df["Price"]

print("Rows:", len(sales_df))
print("Invalid dates:", sales_df["InvoiceDate"].isna().sum())
print("Missing Customer IDs:", sales_df["Customer ID"].isna().sum())
print("Total recorded revenue:", round(sales_df["Revenue"].sum(), 2))

display(sales_df.head())

In [ ]:
print("Missing descriptions:", sales_df["Description"].isna().sum())
print("Zero-price rows:", (sales_df["Price"] == 0).sum())
print("Negative-price rows:", (sales_df["Price"] < 0).sum())
print("Negative-quantity rows:", (sales_df["Quantity"] < 0).sum())
print("Duplicate rows:", sales_df.duplicated().sum())

print("\nData types:")
print(sales_df.dtypes)

In [ ]:

sales_df["Is_Return"] = sales_df["Quantity"] < 0
sales_df["Is_Zero_Price"] = sales_df["Price"] == 0
sales_df["Missing_Description"] = sales_df["Description"].isna()

final_file = processed_folder / "retail_sales_final.csv"

sales_df.to_csv(final_file, index=False, encoding="utf-8-sig")

print("Final dataset saved!")
print("Rows:", len(sales_df))
print("Returns flagged:", sales_df["Is_Return"].sum())
print("Zero-price rows flagged:", sales_df["Is_Zero_Price"].sum())
print("Missing descriptions flagged:", sales_df["Missing_Description"].sum())
print("File:", final_file.resolve())


In [ ]:
# Monthly sales analysis
sales_df["Month"] = sales_df["InvoiceDate"].dt.to_period("M")

monthly_sales = (
    sales_df.groupby("Month")["Revenue"]
    .sum()
    .reset_index()
)

monthly_sales["Month"] = monthly_sales["Month"].astype(str)

display(monthly_sales.head(12))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    monthly_sales["Month"],
    monthly_sales["Revenue"],
    marker="o"
)

plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Net Recorded Transaction Revenue")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Check missing values in the sales dataset

missing_report = pd.DataFrame({
    "Missing_Count": sales_df.isna().sum(),
    "Missing_Percentage": (
        sales_df.isna().sum() / len(sales_df) * 100
    ).round(2)
})

display(missing_report)

In [ ]:
# Check whether missing descriptions can be recovered
# from other rows with the same StockCode

description_lookup = (
    sales_df.dropna(subset=["Description"])
    .groupby("StockCode")["Description"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
)

missing_desc_mask = sales_df["Description"].isna()

recoverable_mask = (
    missing_desc_mask
    & sales_df["StockCode"].isin(description_lookup.index)
)

print("Missing descriptions:", missing_desc_mask.sum())
print("Potentially recoverable:", recoverable_mask.sum())
print(
    "Not recoverable by StockCode:",
    missing_desc_mask.sum() - recoverable_mask.sum()
)

In [ ]:
# Check product codes associated with multiple descriptions

description_counts = (
    sales_df.dropna(subset=["Description"])
    .groupby("StockCode")["Description"]
    .nunique()
)

conflicting_codes = description_counts[description_counts > 1]

print("Unique product codes:", sales_df["StockCode"].nunique())
print("Codes with multiple descriptions:", len(conflicting_codes))

display(conflicting_codes.head(10))

In [ ]:
# Display examples of product codes with multiple descriptions

for code in conflicting_codes.head(5).index:
    descriptions = (
        sales_df.loc[
            sales_df["StockCode"] == code, "Description"
        ]
        .dropna()
        .unique()
    )

    print(f"\nStockCode: {code}")
    for description in descriptions:
        print(" -", description)

In [ ]:
# Inspect all rows for the first five conflicting product codes

for code in conflicting_codes.head(5).index:
    print(f"\n--- StockCode: {code} ---")

    display(
        sales_df.loc[
            sales_df["StockCode"] == code,
            ["Invoice", "StockCode", "Description",
             "Quantity", "InvoiceDate", "Price"]
        ]
        .drop_duplicates()
        .head(10)
    )

In [ ]:
display(
    sales_df.loc[
        sales_df["StockCode"] == "10120",
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price"]
    ]
    .drop_duplicates()
    .head(15)
)

In [ ]:
code = "10120"

print("Rows for this StockCode:",
      (sales_df["StockCode"] == code).sum())

display(
    sales_df.loc[
        sales_df["StockCode"] == code,
        "Description"
    ].value_counts(dropna=False)
)

In [ ]:
# Count missing descriptions that have exactly one
# distinct non-missing description for their StockCode

missing_mask = sales_df["Description"].isna()

recoverable_codes = description_counts[
    description_counts == 1
].index

safe_candidates = (
    missing_mask
    & sales_df["StockCode"].isin(recoverable_codes)
)

print("Missing descriptions:", missing_mask.sum())
print("Potentially safe to recover:", safe_candidates.sum())
print(
    "Need further investigation:",
    missing_mask.sum() - safe_candidates.sum()
)

In [ ]:
# Collect product descriptions that appear unique per StockCode

safe_codes = description_counts[
    description_counts == 1
].index

safe_description_lookup = (
    sales_df.dropna(subset=["Description"])
    .loc[lambda df: df["StockCode"].isin(safe_codes)]
    .drop_duplicates(subset=["StockCode", "Description"])
    .set_index("StockCode")["Description"]
)

print("Candidate product codes:", len(safe_description_lookup))

display(safe_description_lookup.head(20))

In [ ]:
# Review potentially suspicious descriptions before filling missing values

suspicious_pattern = (
    r"damaged|damage|check|error|return|wet|rusty|"
    r"adjust|debt|broken|missing|wrong|invc|"
    r"crack|faulty|unsaleable"
)

suspicious_descriptions = safe_description_lookup[
    safe_description_lookup.str.contains(
        suspicious_pattern,
        case=False,
        na=False,
        regex=True
    )
]

print("Candidate product codes:", len(safe_description_lookup))
print("Potentially suspicious descriptions:", len(suspicious_descriptions))

display(suspicious_descriptions.head(30))

In [ ]:
# Display all suspicious descriptions found by our screening rule
pd.set_option("display.max_rows", None)

display(suspicious_descriptions)

pd.reset_option("display.max_rows")

In [ ]:
# Create a copy so the current sales_df remains unchanged
sales_clean = sales_df.copy()

# Build a lookup of codes with exactly one distinct description
safe_codes = description_counts[description_counts == 1].index

safe_lookup = safe_description_lookup[
    ~safe_description_lookup.index.isin(
        suspicious_descriptions.index
    )
]

# Fill only missing descriptions with a reliable lookup
missing_mask = sales_clean["Description"].isna()

sales_clean["Description"] = sales_clean["Description"].fillna(
    sales_clean["StockCode"].map(safe_lookup)
)

filled_count = (
    missing_mask & sales_clean["Description"].notna()
).sum()

print("Descriptions filled:", filled_count)
print("Descriptions still missing:", sales_clean["Description"].isna().sum())

In [ ]:
print("Total rows:", len(sales_clean))
print("Missing descriptions:", sales_clean["Description"].isna().sum())
print("Missing Customer IDs:", sales_clean["Customer ID"].isna().sum())
print("Duplicate rows:", sales_clean.duplicated().sum())

In [ ]:
# Quantity data-quality audit

print("Negative quantities:", (sales_clean["Quantity"] < 0).sum())
print("Zero quantities:", (sales_clean["Quantity"] == 0).sum())
print("Positive quantities:", (sales_clean["Quantity"] > 0).sum())

print("\nQuantity statistics:")
display(sales_clean["Quantity"].describe())

print("\nLargest 10 quantities:")
display(
    sales_clean.nlargest(10, "Quantity")[
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price"]
    ]
)


In [ ]:

# Step 3: Inspect negative-quantity transactions

negative_qty = sales_clean[sales_clean["Quantity"] < 0]

print("Total negative-quantity rows:", len(negative_qty))

display(
    negative_qty[
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price", "Country"]
    ].head(15)
)


In [ ]:

# Step 4: Inspect transactions with zero price

zero_price = sales_clean[sales_clean["Price"] == 0]

print("Total zero-price rows:", len(zero_price))
print("Negative quantities:", (zero_price["Quantity"] < 0).sum())
print("Positive quantities:", (zero_price["Quantity"] > 0).sum())

display(
    zero_price[
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price", "Country"]
    ].head(15)
)


In [ ]:

# Step 5: Find the most common descriptions in zero-price rows

print("Top zero-price descriptions:")

display(
    zero_price["Description"]
    .fillna("Missing Description")
    .value_counts()
    .head(20)
    .rename_axis("Description")
    .reset_index(name="Count")
)


In [ ]:

# Step 6: Inspect zero-price rows with positive quantities
# and product-like descriptions

product_like_zero = zero_price[
    (zero_price["Quantity"] > 0) &
    zero_price["Description"].notna()
]

print("Positive-quantity zero-price rows with descriptions:",
      len(product_like_zero))

display(
    product_like_zero[
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Country"]
    ].head(20)
)


In [ ]:

# Step 7: Flag unusual records without deleting them

sales_clean["Is_Zero_Price"] = sales_clean["Price"] == 0
sales_clean["Is_Return"] = sales_clean["Quantity"] < 0
sales_clean["Missing_Description"] = sales_clean["Description"].isna()

print("Zero-price rows:", sales_clean["Is_Zero_Price"].sum())
print("Return/negative-quantity rows:", sales_clean["Is_Return"].sum())
print("Missing descriptions:", sales_clean["Missing_Description"].sum())

print("\nFlag combinations:")
display(
    sales_clean.groupby(
        ["Is_Zero_Price", "Is_Return"]
    ).size().rename("Row_Count").reset_index()
)


In [ ]:

# Step 8: Inspect the largest positive quantities

positive_sales = sales_clean[sales_clean["Quantity"] > 0]

print("Largest quantities:")
display(
    positive_sales.nlargest(15, "Quantity")[
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price", "Country"]
    ]
)


In [ ]:

# Step 9: Summarize unusually large positive quantities

print("Positive quantities greater than 1,000:",
      (positive_sales["Quantity"] > 1000).sum())

print("Positive quantities greater than 10,000:",
      (positive_sales["Quantity"] > 10000).sum())

print("Positive quantity statistics:")
display(positive_sales["Quantity"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
))


In [ ]:

# Step 10: Count unusually large quantities

threshold = positive_sales["Quantity"].quantile(0.99)

extreme_sales = positive_sales[
    positive_sales["Quantity"] > threshold
]

print("99th percentile threshold:", threshold)
print("Rows above the 99th percentile:", len(extreme_sales))

display(
    extreme_sales[
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price", "Country"]
    ]
    .sort_values("Quantity", ascending=False)
    .head(10)
)


In [ ]:

# Step 11: Inspect all items on the two largest-quantity invoices

largest_invoices = ["581483", "541431"]

display(
    sales_clean[
        sales_clean["Invoice"].astype("string").isin(largest_invoices)
    ][
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price", "Country"]
    ].sort_values(["Invoice", "Quantity"], ascending=[True, False])
)


In [ ]:

# Step 12: Validate transaction dates

print("Earliest transaction:", sales_clean["InvoiceDate"].min())
print("Latest transaction:", sales_clean["InvoiceDate"].max())
print("Missing dates:", sales_clean["InvoiceDate"].isna().sum())

daily_counts = (
    sales_clean.assign(
        Date=sales_clean["InvoiceDate"].dt.date
    )
    .groupby("Date")
    .size()
)

print("Number of dates with transactions:", len(daily_counts))
print("Dates with zero transactions are not included in this count.")



In [ ]:

import pandas as pd
from pathlib import Path

processed_folder = Path("../data/processed")
sales_file = processed_folder / "retail_sales_analysis.csv"

sales_clean = pd.read_csv(
    sales_file,
    dtype={"Invoice": "string", "StockCode": "string"}
)

sales_clean["InvoiceDate"] = pd.to_datetime(
    sales_clean["InvoiceDate"], errors="coerce"
)
sales_clean["Quantity"] = pd.to_numeric(
    sales_clean["Quantity"], errors="coerce"
)
sales_clean["Price"] = pd.to_numeric(
    sales_clean["Price"], errors="coerce"
)

# Restore the description improvements from our previous steps
# if you have not saved them to a file, they will need to be reapplied.

print("Dataset loaded successfully!")
print("Rows:", len(sales_clean))
print("Columns:", sales_clean.columns.tolist())


In [ ]:

# Step 13: Restore flags and validate prices and quantities

sales_clean["Is_Zero_Price"] = sales_clean["Price"] == 0
sales_clean["Is_Return"] = sales_clean["Quantity"] < 0
sales_clean["Missing_Description"] = sales_clean["Description"].isna()

print("Negative prices:", (sales_clean["Price"] < 0).sum())
print("Zero quantities:", (sales_clean["Quantity"] == 0).sum())
print("Negative quantities:", (sales_clean["Quantity"] < 0).sum())
print("Zero prices:", (sales_clean["Price"] == 0).sum())

print("\nNegative-price records:")
display(
    sales_clean[sales_clean["Price"] < 0][
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price", "Country"]
    ].head(10)
)



In [ ]:

# Step 14: Missing-value audit

missing_summary = pd.DataFrame({
    "Missing_Count": sales_clean.isna().sum(),
    "Missing_Percentage": (
        sales_clean.isna().sum() / len(sales_clean) * 100
    ).round(2)
})

display(missing_summary)


In [ ]:

# Step 15: Recover descriptions when a StockCode
# has exactly one known description

description_counts = (
    sales_clean.dropna(subset=["Description"])
    .groupby("StockCode")["Description"]
    .nunique()
)

safe_codes = description_counts[
    description_counts == 1
].index

safe_lookup = (
    sales_clean[
        sales_clean["StockCode"].isin(safe_codes)
        & sales_clean["Description"].notna()
    ]
    .drop_duplicates(subset=["StockCode", "Description"])
    .set_index("StockCode")["Description"]
)

missing_before = sales_clean["Description"].isna().sum()

sales_clean["Description"] = sales_clean["Description"].fillna(
    sales_clean["StockCode"].map(safe_lookup)
)

missing_after = sales_clean["Description"].isna().sum()

print("Missing descriptions before:", missing_before)
print("Descriptions recovered:", missing_before - missing_after)
print("Missing descriptions after:", missing_after)


In [ ]:

# Step 16: Find potentially suspicious descriptions
# among rows whose descriptions were recovered

suspicious_pattern = (
    r"damag|check|error|return|wet|rusty|adjust|debt|"
    r"broken|missing|wrong|invc|crack|faulty|unsaleable|"
    r"lost|found|smashed|short"
)

recovered_rows = sales_clean[
    sales_clean["StockCode"].isin(safe_codes)
    & sales_clean["Description"].notna()
]

suspicious_rows = recovered_rows[
    recovered_rows["Description"].str.contains(
        suspicious_pattern, case=False, na=False
    )
]

print("Rows with descriptions needing review:",
      len(suspicious_rows))

display(
    suspicious_rows[
        ["StockCode", "Description"]
    ]
    .drop_duplicates()
    .head(30)
)


In [ ]:

# Step 17: Inspect clearly operational descriptions

operational_pattern = (
    r"^(damages?|lost|found|damaged|missing|smashed|"
    r"short|adjustment by .+|thrown away.*|check|\?)$"
)

operational_descriptions = (
    sales_clean.loc[
        sales_clean["Description"].str.contains(
            operational_pattern, case=False, na=False
        ),
        ["StockCode", "Description"]
    ]
    .drop_duplicates()
    .sort_values(["StockCode", "Description"])
)

print("Potential operational descriptions:",
      len(operational_descriptions))

display(operational_descriptions)


In [ ]:

# Step 18: Create a clean review list

operational_pattern = (
    r"^(?:damages?|lost|found|damaged|missing|smashed|"
    r"short|adjustment by .+|thrown away.*|check|\?)$"
)

operational_descriptions = (
    sales_clean.loc[
        sales_clean["Description"].str.contains(
            operational_pattern, case=False, na=False
        ),
        ["StockCode", "Description"]
    ]
    .drop_duplicates()
    .sort_values(["StockCode", "Description"])
)

display(operational_descriptions.head(30))
print("Total distinct combinations:", len(operational_descriptions))



In [ ]:

# Step 19: Mark operational descriptions as missing
# so they won't be mistaken for product names

operational_pattern = (
    r"^(?:damages?|lost|found|damaged|missing|smashed|"
    r"short|adjustment by .+|thrown away.*|check|\?)$"
)

operational_mask = sales_clean["Description"].str.contains(
    operational_pattern, case=False, na=False
)

print("Rows with operational descriptions:", operational_mask.sum())

sales_clean.loc[operational_mask, "Description"] = pd.NA

print(
    "Missing descriptions after review:",
    sales_clean["Description"].isna().sum()
)


In [ ]:

# Step 20: Final data-quality check

print("Total rows:", len(sales_clean))
print("Duplicate rows:", sales_clean.duplicated().sum())
print("Missing Invoice:", sales_clean["Invoice"].isna().sum())
print("Missing StockCode:", sales_clean["StockCode"].isna().sum())
print("Missing Description:", sales_clean["Description"].isna().sum())
print("Missing InvoiceDate:", sales_clean["InvoiceDate"].isna().sum())
print("Missing Quantity:", sales_clean["Quantity"].isna().sum())
print("Missing Price:", sales_clean["Price"].isna().sum())
print("Missing Customer ID:", sales_clean["Customer ID"].isna().sum())
print("Missing Country:", sales_clean["Country"].isna().sum())


In [ ]:

# Step 21: Refresh flags and calculate revenue

sales_clean["Missing_Description"] = sales_clean["Description"].isna()
sales_clean["Is_Return"] = sales_clean["Quantity"] < 0
sales_clean["Is_Zero_Price"] = sales_clean["Price"] == 0
sales_clean["Revenue"] = sales_clean["Quantity"] * sales_clean["Price"]

print("Rows:", len(sales_clean))
print("Missing-description flags:", sales_clean["Missing_Description"].sum())
print("Return rows:", sales_clean["Is_Return"].sum())
print("Zero-price rows:", sales_clean["Is_Zero_Price"].sum())
print("Revenue column added:", "Revenue" in sales_clean.columns)


In [ ]:

from pathlib import Path

output_path = Path("../data/processed/retail_sales_cleaned_v2.csv")
sales_clean.to_csv(output_path, index=False)

print("Dataset saved successfully!")
print("File:", output_path.resolve())
print("Rows saved:", len(sales_clean))
print("Columns saved:", len(sales_clean.columns))
print("File size (MB):", round(output_path.stat().st_size / (1024 * 1024), 2))


In [ ]:

import pandas as pd

file_path = "../data/processed/retail_sales_cleaned_v2.csv"

check_df = pd.read_csv(file_path)

print("Loaded successfully!")
print("Shape:", check_df.shape)
print("\nColumn names:")
print(check_df.columns.tolist())
print("\nFirst 5 rows:")
display(check_df.head())



In [ ]:

print("Missing Revenue values:", check_df["Revenue"].isna().sum())
print("Fifth row Revenue:", check_df.loc[4, "Revenue"])
print("Missing InvoiceDate values:", check_df["InvoiceDate"].isna().sum())


In [ ]:

# Step 24: Basic retail business overview

print("Date range:")
print(check_df["InvoiceDate"].min(), "to", check_df["InvoiceDate"].max())

print("\nUnique invoices:", check_df["Invoice"].nunique())
print("Unique products:", check_df["StockCode"].nunique())
print("Unique countries:", check_df["Country"].nunique())
print("Unique customers:", check_df["Customer ID"].nunique())

print("\nTotal quantity:", check_df["Quantity"].sum())
print("Total revenue, including returns:", round(check_df["Revenue"].sum(), 2))

print("\nTop 10 countries by revenue:")
print(
    check_df.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .round(2)
)


In [ ]:

# Step 25: Monthly revenue trend

check_df["InvoiceDate"] = pd.to_datetime(check_df["InvoiceDate"])

check_df["YearMonth"] = check_df["InvoiceDate"].dt.to_period("M").astype(str)

monthly_revenue = (
    check_df.groupby("YearMonth")["Revenue"]
    .sum()
    .reset_index()
)

print("Monthly revenue:")
display(monthly_revenue)

print("\nHighest-revenue month:")
display(monthly_revenue.loc[monthly_revenue["Revenue"].idxmax()])

print("\nLowest-revenue month:")
display(monthly_revenue.loc[monthly_revenue["Revenue"].idxmin()])


In [ ]:

# Step 24: Investigate missing product descriptions

missing_desc = check_df[check_df["Description"].isna()]

print("Rows with missing descriptions:", len(missing_desc))
print("Unique stock codes with missing descriptions:",
      missing_desc["StockCode"].nunique())

print("\nMost frequent stock codes with missing descriptions:")
display(
    missing_desc["StockCode"]
    .value_counts()
    .head(15)
    .rename_axis("StockCode")
    .reset_index(name="MissingDescriptionRows")
)


In [ ]:

# Step 25: Find stock codes with a single known description

known_descriptions = check_df.dropna(subset=["Description"])

description_counts = (
    known_descriptions.groupby("StockCode")["Description"].nunique()
)

safe_codes = description_counts[description_counts == 1].index

safe_lookup = (
    known_descriptions[known_descriptions["StockCode"].isin(safe_codes)]
    .drop_duplicates(subset=["StockCode", "Description"])
    .set_index("StockCode")["Description"]
)

missing_mask = check_df["Description"].isna()
recoverable_mask = missing_mask & check_df["StockCode"].isin(safe_lookup.index)

print("Missing-description rows:", missing_mask.sum())
print("Rows with a possible safe recovery:", recoverable_mask.sum())
print("Rows still needing review:", (missing_mask & ~recoverable_mask).sum())


In [ ]:

# Step 26: Preview descriptions before applying recovery

recovery_preview = check_df.loc[
    recoverable_mask, ["StockCode"]
].copy()

recovery_preview["RecoveredDescription"] = (
    recovery_preview["StockCode"].map(safe_lookup)
)

display(
    recovery_preview.drop_duplicates()
    .head(30)
)

print(
    "Unique stock codes proposed for recovery:",
    recovery_preview["StockCode"].nunique()
)


In [ ]:

# Step 27: Check stock codes and their mapped descriptions

display(
    safe_lookup.rename_axis("StockCode")
    .reset_index(name="KnownDescription")
    .head(30)
)


In [ ]:

# Step 28: Inspect missing rows and their proposed descriptions

recovery_check = check_df.loc[
    recoverable_mask, ["StockCode", "Description"]
].copy()

recovery_check["ProposedDescription"] = (
    recovery_check["StockCode"].map(safe_lookup)
)

display(recovery_check.head(20))

print(
    "Missing descriptions eligible for recovery:",
    recoverable_mask.sum()
)
print(
    "Proposed descriptions missing:",
    recovery_check["ProposedDescription"].isna().sum()
)


In [ ]:

print("StockCode dtype:", check_df["StockCode"].dtype)
print("Lookup index dtype:", safe_lookup.index.dtype)

print("\nOriginal missing row:")
print(check_df.loc[recoverable_mask, ["StockCode", "Description"]].iloc[0].to_dict())

first_code = check_df.loc[recoverable_mask, "StockCode"].iloc[0]

print("\nFirst stock code:", repr(first_code))
print("Lookup result:", repr(safe_lookup.get(first_code)))
print("Is code in lookup:", first_code in safe_lookup.index)


In [ ]:

# Step 30: Verify recovery mapping

recovery_check = check_df.loc[
    recoverable_mask, ["StockCode", "Description"]
].copy()

recovery_check["ProposedDescription"] = (
    recovery_check["StockCode"].map(safe_lookup)
)

print("Eligible rows:", len(recovery_check))
print(
    "Proposed descriptions missing:",
    recovery_check["ProposedDescription"].isna().sum()
)

print("\nSample mappings:")
for _, row in recovery_check.head(10).iterrows():
    print(
        repr(row["StockCode"]),
        "->",
        repr(row["ProposedDescription"])
    )


In [ ]:

# Step 31: Apply verified description recovery

missing_before = check_df["Description"].isna().sum()

check_df["Description"] = check_df["Description"].fillna(
    check_df["StockCode"].map(safe_lookup)
)

missing_after = check_df["Description"].isna().sum()

print("Missing before:", missing_before)
print("Descriptions recovered:", missing_before - missing_after)
print("Missing remaining:", missing_after)
print("Total rows:", len(check_df))


In [ ]:

# Step 32: Inspect stock codes still missing descriptions

remaining_missing = check_df[check_df["Description"].isna()]

print("Rows still missing descriptions:", len(remaining_missing))
print(
    "Unique stock codes:",
    remaining_missing["StockCode"].nunique()
)

print("\nMost frequent stock codes:")
display(
    remaining_missing["StockCode"]
    .value_counts()
    .head(20)
    .rename_axis("StockCode")
    .reset_index(name="MissingRows")
)


In [ ]:

# Step 33: Inspect remaining missing-description records

display(
    remaining_missing[
        ["Invoice", "StockCode", "Quantity", "InvoiceDate",
         "Price", "Customer ID", "Country"]
    ].head(20)
)

print("\nStock codes with missing descriptions:")
print(remaining_missing["StockCode"].value_counts().head(10))


In [ ]:

# Step 34: Check whether any remaining descriptions can be safely recovered

known_descriptions = check_df.dropna(subset=["Description"])

description_counts = (
    known_descriptions.groupby("StockCode")["Description"].nunique()
)

safe_codes = description_counts[description_counts == 1].index

safe_lookup = (
    known_descriptions[known_descriptions["StockCode"].isin(safe_codes)]
    .drop_duplicates(subset=["StockCode", "Description"])
    .set_index("StockCode")["Description"]
)

remaining_missing = check_df[check_df["Description"].isna()]

still_recoverable = remaining_missing[
    remaining_missing["StockCode"].isin(safe_lookup.index)
]

print("Remaining missing-description rows:", len(remaining_missing))
print("Rows safely recoverable now:", len(still_recoverable))
print("Unique stock codes recoverable:", still_recoverable["StockCode"].nunique())



In [ ]:

# Step 35: Update the missing-description flag

check_df["Missing_Description"] = check_df["Description"].isna()

print("Total rows:", len(check_df))
print("Missing descriptions:", check_df["Description"].isna().sum())
print("Flagged as missing:", check_df["Missing_Description"].sum())
print(
    "Flag consistency:",
    (check_df["Missing_Description"] == check_df["Description"].isna()).all()
)


In [ ]:

# Step 36: Validate cleaning flags

print("Negative-quantity rows:", (check_df["Quantity"] < 0).sum())
print("Rows flagged as returns:", check_df["Is_Return"].sum())
print(
    "Return flag consistent:",
    (check_df["Is_Return"] == (check_df["Quantity"] < 0)).all()
)

print("\nZero-price rows:", (check_df["Price"] == 0).sum())
print("Rows flagged as zero price:", check_df["Is_Zero_Price"].sum())
print(
    "Zero-price flag consistent:",
    (check_df["Is_Zero_Price"] == (check_df["Price"] == 0)).all()
)


In [ ]:

# Step 37: Validate revenue calculations

expected_revenue = check_df["Quantity"] * check_df["Price"]

revenue_mismatch = (
    (check_df["Revenue"] - expected_revenue).abs() > 0.000001
)

print("Missing revenue values:", check_df["Revenue"].isna().sum())
print("Revenue mismatches:", revenue_mismatch.sum())
print("Revenue calculation consistent:", (~revenue_mismatch).all())
print("Total revenue:", round(check_df["Revenue"].sum(), 2))


In [ ]:

# Step 38: Final data-type and invalid-value checks

print("Data types:")
display(check_df.dtypes.to_frame(name="DataType"))

print("\nInvalid-value checks:")
print("Missing invoices:", check_df["Invoice"].isna().sum())
print("Missing stock codes:", check_df["StockCode"].isna().sum())
print("Missing dates:", check_df["InvoiceDate"].isna().sum())
print("Missing quantities:", check_df["Quantity"].isna().sum())
print("Missing prices:", check_df["Price"].isna().sum())
print("Negative prices:", (check_df["Price"] < 0).sum())
print("Zero quantities:", (check_df["Quantity"] == 0).sum())
print("Duplicate rows:", check_df.duplicated().sum())


In [ ]:

# Step 39: Final invalid-value checks

print("Missing invoices:", check_df["Invoice"].isna().sum())
print("Missing stock codes:", check_df["StockCode"].isna().sum())
print("Missing dates:", check_df["InvoiceDate"].isna().sum())
print("Missing quantities:", check_df["Quantity"].isna().sum())
print("Missing prices:", check_df["Price"].isna().sum())
print("Negative prices:", (check_df["Price"] < 0).sum())
print("Zero quantities:", (check_df["Quantity"] == 0).sum())
print("Duplicate rows:", check_df.duplicated().sum())



In [ ]:

from pathlib import Path

# Exclude temporary analysis columns such as YearMonth
final_df = check_df.drop(
    columns=["YearMonth"],
    errors="ignore"
).copy()

# Refresh the missing-description flag
final_df["Missing_Description"] = final_df["Description"].isna()

# Save as a new file; preserve all existing checkpoints
output_path = Path("../data/processed/retail_sales_cleaned_final.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

final_df.to_csv(output_path, index=False)

print("Final cleaned dataset saved successfully!")
print("File:", output_path.resolve())
print("Rows:", len(final_df))
print("Columns:", len(final_df.columns))
print("File size (MB):", round(output_path.stat().st_size / (1024 * 1024), 2))
print("\nFinal columns:")
print(final_df.columns.tolist())


In [ ]:

# Step 41: Verify the saved final dataset

from pathlib import Path
import pandas as pd

output_path = Path("../data/processed/retail_sales_cleaned_final.csv")

saved_df = pd.read_csv(
    output_path,
    dtype={"Invoice": "string", "StockCode": "string"},
    parse_dates=["InvoiceDate"]
)

print("Saved file exists:", output_path.exists())
print("Saved shape:", saved_df.shape)
print("Duplicate rows:", saved_df.duplicated().sum())
print("Missing descriptions:", saved_df["Description"].isna().sum())
print(
    "Missing-description flag consistent:",
    (saved_df["Missing_Description"] == saved_df["Description"].isna()).all()
)
print(
    "Revenue consistent:",
    (
        saved_df["Revenue"]
        - saved_df["Quantity"] * saved_df["Price"]
    ).abs().le(0.000001).all()
)


In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
file_path = project_root / "data" / "processed" / "retail_sales_final.csv"

df = pd.read_csv(file_path)

print("DATA QUALITY VALIDATION")
print("-" * 35)
print("File exists:", file_path.exists())
print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values:")
print(df.isna().sum())
print("\nData types:")
print(df.dtypes)
print("\nDate range:")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
print(df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print("Invalid dates:", df["InvoiceDate"].isna().sum())
print("\nNegative prices:", (df["Price"] < 0).sum())
print("Negative quantities:", (df["Quantity"] < 0).sum())

In [ ]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

print("InvoiceDate type:", df["InvoiceDate"].dtype)
print("Invalid dates:", df["InvoiceDate"].isna().sum())

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
raw_file = project_root / "data" / "processed" / "retail_sales_final.csv"

print("ETL PIPELINE CHECK")
print("=" * 40)

# 1. Check whether the final dataset exists
print("Final dataset exists:", raw_file.exists())

if raw_file.exists():
    df = pd.read_csv(raw_file)

    # 2. Basic data-quality checks
    checks = {
        "Dataset is not empty": len(df) > 0,
        "Required columns exist": all(
            col in df.columns
            for col in [
                "Invoice", "StockCode", "Quantity",
                "InvoiceDate", "Price", "Revenue"
            ]
        ),
        "No duplicate rows": not df.duplicated().any(),
        "No invalid invoice dates": pd.to_datetime(
            df["InvoiceDate"], errors="coerce"
        ).notna().all(),
        "No negative prices": (df["Price"] >= 0).all(),
        "Return flag exists": "Is_Return" in df.columns,
        "Missing-description flag exists": "Missing_Description" in df.columns,
    }

    for check, passed in checks.items():
        print(f"{'PASS' if passed else 'CHECK'}: {check}")

    print("\nRows:", len(df))
    print("Columns:", len(df.columns))
    print("\nETL validation completed.")

In [ ]:
print("DATA QUALITY FLAG CHECK")
print("=" * 40)

print(
    "Missing descriptions flagged:",
    df["Missing_Description"].sum()
)
print(
    "Actual missing descriptions:",
    df["Description"].isna().sum()
)

print(
    "Returns flagged:",
    df["Is_Return"].sum()
)
print(
    "Negative quantities:",
    (df["Quantity"] < 0).sum()
)

print(
    "Zero-price transactions flagged:",
    df["Is_Zero_Price"].sum()
)
print(
    "Actual zero prices:",
    (df["Price"] == 0).sum()
)

In [ ]:
from pathlib import Path
import pandas as pd

file_path = Path(r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_final.csv")

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nFirst 5 descriptions:")
print(df["Description"].head().to_string(index=False))

In [ ]:
import re

# Keep a backup of the original descriptions
df["Description_original"] = df["Description"].copy()

def clean_description(description):
    if pd.isna(description):
        return description

    description = str(description).upper().strip()

    # Remove measurements such as 15CM
    description = re.sub(r"\b\d+(?:\.\d+)?\s*(?:CM|MM|M|INCHES|INCH)\b", "", description)

    # Remove pack/count wording
    description = re.sub(
        r"\bPACK\s+OF\s+\d+\b|\b\d+\s*(?:PACK|PCS|PIECES)\b",
        "",
        description
    )

    # Remove colour words
    colours = [
        "PINK", "WHITE", "BLACK", "RED", "BLUE",
        "GREEN", "YELLOW", "ORANGE", "PURPLE",
        "BROWN", "GREY", "GRAY", "CREAM"
    ]

    for colour in colours:
        description = re.sub(rf"\b{colour}\b", "", description)

    # Remove remaining standalone numbers
    description = re.sub(r"\b\d+(?:\.\d+)?\b", "", description)

    # Clean up extra spaces and punctuation at the ends
    description = re.sub(r"\s+", " ", description).strip()
    description = description.strip(' "\'-')

    return description


# Test on the first five rows only
sample = df["Description"].head(5)

for original in sample:
    print("Original :", original)
    print("Processed:", clean_description(original))
    print("-" * 50)

In [ ]:
import re
import pandas as pd

def clean_description(description):
    if pd.isna(description):
        return description

    description = str(description).upper().strip()

    # Remove measurements such as 15CM, 7" and 2.5MM
    description = re.sub(
        r'\b\d+(?:\.\d+)?\s*(?:CM|MM|M|INCHES|INCH|")',
        '',
        description
    )

    # Remove pack/count wording
    description = re.sub(
        r'\bPACK\s+OF\s+\d+\b|\b\d+\s*(?:PACK|PCS|PIECES)\b',
        '',
        description
    )

    # Remove selected colour words
    colours = [
        "PINK", "WHITE", "BLACK", "RED", "BLUE",
        "GREEN", "YELLOW", "ORANGE", "PURPLE",
        "BROWN", "GREY", "GRAY", "CREAM"
    ]

    for colour in colours:
        description = re.sub(rf'\b{colour}\b', '', description)

    # Remove remaining standalone numbers
    description = re.sub(r'\b\d+(?:\.\d+)?\b', '', description)

    # Remove leftover quotation marks and tidy spaces
    description = re.sub(r'["“”]', '', description)
    description = re.sub(r'\s+', ' ', description).strip()
    description = description.strip(" -'")

    return description

In [ ]:
for original in df["Description"].head(5):
    print("Original :", original)
    print("Processed:", clean_description(original))
    print("-" * 50)

In [ ]:
from pathlib import Path

# Define output paths
output_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_description_cleaned.csv"
)

backup_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_description_backup.csv"
)

# Save the original descriptions as a separate backup
df[["Description_original"]].to_csv(backup_path, index=False)

# Clean only the Description column
df["Description"] = df["Description"].apply(clean_description)

# Save the cleaned dataset to a new file
df.drop(columns=["Description_original"]).to_csv(output_path, index=False)

print("Description cleaning completed!")
print("Original rows:", len(df))
print("Cleaned file:", output_path)
print("Backup file:", backup_path)

In [ ]:
print("Dataset shape:", df.shape)

print("\nFirst 10 cleaned descriptions:")
print(df["Description"].head(10).to_string(index=False))

print("\nMissing descriptions:", df["Description"].isna().sum())
print("Empty descriptions:", df["Description"].eq("").sum())

print(
    "Descriptions still containing digits:",
    df["Description"].str.contains(r"\d", na=False).sum()
)


In [ ]:
# 1. Check descriptions that became empty
empty_rows = df[
    df["Description"].fillna("").str.strip().eq("")
]

print("Rows with empty cleaned descriptions:", len(empty_rows))

if not empty_rows.empty:
    print("\nOriginal descriptions that became empty:")
    print(empty_rows[["Description_original", "Description"]].to_string(index=False))


# 2. Inspect descriptions that still contain digits
digit_rows = df[
    df["Description"].str.contains(r"\d", na=False)
]

print("\nRows whose cleaned descriptions still contain digits:", len(digit_rows))

print("\nExamples to review:")
print(
    digit_rows[["Description_original", "Description"]]
    .drop_duplicates()
    .head(30)
    .to_string(index=False)
)

In [ ]:
import pandas as pd
import re
from pathlib import Path

# Use the original descriptions kept in your notebook, if available
source = (
    df["Description_original"]
    if "Description_original" in df.columns
    else df["Description"]
)

def clean_description(text):
    if pd.isna(text):
        return pd.NA

    text = str(text).upper().strip()

    # Remove measurements such as 15CM, 500G and 7"
    text = re.sub(
        r'\b\d+(?:\.\d+)?\s*(?:CM|MM|KG|G|GRAMS?|ML|L|INCHES|INCH)\b',
        ' ', text
    )
    text = re.sub(r'\b\d+(?:\.\d+)?\s*"', ' ', text)

    # Remove dimensions such as 40X40CM
    text = re.sub(
        r'\b\d+\s*[X×]\s*\d+\s*(?:CM|MM)?\b',
        ' ', text
    )

    # Remove pack wording
    text = re.sub(r'\bPACK\s+OF\s+\d+\b', ' ', text)

    # Remove selected colour words
    colours = [
        "PINK", "WHITE", "BLACK", "RED", "BLUE",
        "GREEN", "YELLOW", "ORANGE", "PURPLE",
        "BROWN", "GREY", "GRAY", "CREAM"
    ]

    for colour in colours:
        text = re.sub(rf'\b{colour}\b', ' ', text)

    # Remove remaining numbers and unnecessary size wording
    text = re.sub(r'\bSINGLE\s+SIZE\b|\bSIZE\b', ' ', text)
    text = re.sub(r'\d+', ' ', text)

    # Clean extra spaces and quotation marks
    text = re.sub(r'["“”]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip(" -/'")

    return text if text else pd.NA


# Update the Description column directly
df["Description"] = source.apply(clean_description)

# Remove temporary working columns, if present
df.drop(
    columns=["Description_original", "Description_test"],
    errors="ignore",
    inplace=True
)

# Save as a NEW file; do not overwrite your existing CSV
output_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_description_cleaned_v2.csv"
)

df.to_csv(output_path, index=False)

print("Description cleaning completed!")
print("Dataset shape:", df.shape)
print("\nFirst 10 cleaned descriptions:")
print(df["Description"].head(10).to_string(index=False))
print("\nSaved file:", output_path)

In [ ]:

# Validate the cleaned Description column

print("Dataset shape:", df.shape)

print("\nMissing descriptions:",
      df["Description"].isna().sum())

print("Empty descriptions:",
      (df["Description"].notna()
       & df["Description"].str.strip().eq("")).sum())

print("Descriptions still containing digits:",
      df["Description"].str.contains(r"\d", na=False).sum())

print("\nFirst 20 descriptions:")
print(df["Description"].head(20).to_string(index=False))

print("\nUnique product descriptions:",
      df["Description"].nunique())

In [ ]:
from pathlib import Path
import pandas as pd

backup_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_description_backup.csv"
)

backup = pd.read_csv(backup_path)
original_descriptions = backup.iloc[:, 0]

# Find descriptions that had text originally but are now missing
newly_missing = (
    df["Description"].isna()
    & original_descriptions.notna()
)

print("Descriptions removed completely:", newly_missing.sum())

print("\nOriginal descriptions to review:")
print(
    original_descriptions[newly_missing].to_string(index=True)
)

In [ ]:
# Check whether these four rows have descriptions elsewhere
rows_to_check = [6693, 264955, 264956, 895697]

for row_index in rows_to_check:
    stock_code = df.loc[row_index, "StockCode"]

    matches = df.loc[
        (df["StockCode"] == stock_code)
        & df["Description"].notna(),
        "Description"
    ]

    print(f"\nRow index: {row_index}")
    print(f"StockCode: {stock_code}")

    if matches.empty:
        print("No matching product description found.")
    else:
        print("Possible descriptions:")
        print(matches.value_counts().head(5).to_string())

In [ ]:
# Restore the four descriptions using matching product records

repairs = {
    6693: "ROTATING SILVER ANGELS T-LIGHT HLDR",
    264955: "GUMBALL COAT RACK",
    264956: "GUMBALL MONOCHROME COAT RACK",
    895697: "JUMBO BAG VINTAGE CHRISTMAS"
}

for row_index, description in repairs.items():
    df.loc[row_index, "Description"] = description

# Verify the four repaired descriptions
print(df.loc[list(repairs.keys()), ["StockCode", "Description"]])

# Check the remaining missing and empty descriptions
print("\nMissing descriptions:", df["Description"].isna().sum())

print(
    "Empty descriptions:",
    (df["Description"].notna() &
     df["Description"].str.strip().eq("")).sum()
)

# Save a new final file, preserving earlier files
from pathlib import Path

output_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_description_cleaned_final.csv"
)

df.to_csv(output_path, index=False)

print("\nFinal cleaned dataset saved to:", output_path)
print("Dataset shape:", df.shape)


In [ ]:
from pathlib import Path
import pandas as pd

final_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_description_cleaned_final.csv"
)

# Read the saved file for verification
final_df = pd.read_csv(final_path)

print("Saved dataset shape:", final_df.shape)
print("Missing descriptions:", final_df["Description"].isna().sum())
print(
    "Empty descriptions:",
    (final_df["Description"].notna()
     & final_df["Description"].str.strip().eq("")).sum()
)
print(
    "Descriptions containing digits:",
    final_df["Description"].str.contains(r"\d", na=False).sum()
)

In [ ]:
import pandas as pd
from pathlib import Path

# Work on a copy so your current dataframe stays unchanged
df_filled = df.copy()

# Count missing descriptions before recovery
missing_before = df_filled["Description"].isna().sum()

# Ignore obvious data-entry error descriptions
bad_pattern = (
    r"WRONG CODE|WRONG BARCODE|WRONGLY CODED|"
    r"WRONGLY MARKED|WRONGLY SOLD|SOLD AS|CAME CODED AS"
)

# Use rows with valid descriptions as recovery candidates
candidates = df_filled[
    df_filled["StockCode"].notna()
    & df_filled["Description"].notna()
    & ~df_filled["Description"].str.contains(
        bad_pattern, case=False, na=False, regex=True
    )
].copy()

# Find the most frequently used description for each StockCode
description_counts = (
    candidates.groupby(["StockCode", "Description"])
    .size()
    .reset_index(name="frequency")
    .sort_values(
        ["StockCode", "frequency"],
        ascending=[True, False]
    )
)

best_description = (
    description_counts
    .drop_duplicates(subset="StockCode")
    .set_index("StockCode")["Description"]
)

# Fill only missing descriptions with a matching StockCode
missing_mask = df_filled["Description"].isna()

df_filled.loc[missing_mask, "Description"] = (
    df_filled.loc[missing_mask, "StockCode"]
    .map(best_description)
)

# Report the results
missing_after = df_filled["Description"].isna().sum()
recovered = missing_before - missing_after

print("Missing before:", missing_before)
print("Descriptions recovered:", recovered)
print("Missing remaining:", missing_after)

# Save to a new file without overwriting your existing dataset
output_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_description_recovered.csv"
)

df_filled.to_csv(output_path, index=False)

print("\nDataset shape:", df_filled.shape)
print("Saved file:", output_path)

In [ ]:
# Find rows whose descriptions are still missing
remaining = df_filled[df_filled["Description"].isna()].copy()

print("Rows still missing descriptions:", len(remaining))
print("Unique StockCodes:", remaining["StockCode"].nunique())

# Check whether those StockCodes have any other descriptions
remaining_codes = remaining["StockCode"].dropna().unique()

related = df.loc[
    df["StockCode"].isin(remaining_codes)
    & df["Description"].notna(),
    ["StockCode", "Description"]
].drop_duplicates()

print("\nOther descriptions found for these StockCodes:")
print(related.head(30).to_string(index=False))

print("\nRemaining missing rows — first 20:")
print(
    remaining[["StockCode", "InvoiceDate", "Quantity", "Price"]]
    .head(20)
    .to_string(index=False)
)

In [ ]:
from pathlib import Path

# Extract only rows whose descriptions could not be recovered
remaining = df_filled[df_filled["Description"].isna()].copy()

# Keep useful columns that exist in the dataset
review_columns = [
    col for col in [
        "Invoice", "StockCode", "InvoiceDate",
        "Quantity", "Price", "Country", "Revenue"
    ]
    if col in remaining.columns
]

review_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_missing_descriptions_review.csv"
)

remaining[review_columns].to_csv(review_path, index=False)

print("Records saved for review:", len(remaining))
print("Unique missing StockCodes:", remaining["StockCode"].nunique())
print("Review file:", review_path)




In [ ]:
import pandas as pd
from pathlib import Path

file_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_description_recovered.csv"
)

# Load the saved dataset for verification
final_df = pd.read_csv(file_path)

print("Dataset shape:", final_df.shape)
print("Missing descriptions:", final_df["Description"].isna().sum())

print(
    "Empty descriptions:",
    (
        final_df["Description"].notna()
        & final_df["Description"].str.strip().eq("")
    ).sum()
)

print(
    "Descriptions containing digits:",
    final_df["Description"].str.contains(r"\d", na=False).sum()
)

In [ ]:
import pandas as pd
from pathlib import Path

# Load the recovered dataset
input_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_description_recovered.csv"
)

df_final = pd.read_csv(input_path)

# Count rows before removal
rows_before = len(df_final)

# Treat empty or whitespace-only descriptions as missing
df_final["Description"] = df_final["Description"].replace(
    r"^\s*$", pd.NA, regex=True
)

# Remove rows with missing descriptions
df_final = df_final.dropna(subset=["Description"]).copy()

# Save as a new file
output_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_description_final.csv"
)

df_final.to_csv(output_path, index=False)

print("Rows before:", rows_before)
print("Rows removed:", rows_before - len(df_final))
print("Rows remaining:", len(df_final))
print("Dataset shape:", df_final.shape)
print("Missing descriptions:", df_final["Description"].isna().sum())
print("Saved file:", output_path)

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

# Processed data folder
folder = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed"
)

# Existing master file
master_file = folder / "retail_sales_final.csv"

# Verified cleaned dataset
cleaned_file = folder / "retail_sales_description_final.csv"

# Backup of the existing master file
backup_file = folder / "retail_sales_final_before_description_update.csv"

# Check that the required files exist
if not master_file.exists():
    raise FileNotFoundError(f"Master file not found: {master_file}")

if not cleaned_file.exists():
    raise FileNotFoundError(f"Cleaned file not found: {cleaned_file}")

# Verify the cleaned dataset before replacing anything
cleaned_df = pd.read_csv(cleaned_file)

if len(cleaned_df) != 1_013_555:
    raise ValueError("Unexpected row count. No files were replaced.")

if cleaned_df["Description"].isna().any():
    raise ValueError("Missing descriptions remain. No files were replaced.")

if cleaned_df.shape[1] != 12:
    raise ValueError("Unexpected number of columns. No files were replaced.")

# Back up the existing master file
shutil.copy2(master_file, backup_file)

# Update the master file with the verified cleaned dataset
shutil.copy2(cleaned_file, master_file)

# Verify the updated master file
updated_df = pd.read_csv(master_file)

print("Master dataset updated successfully!")
print("File:", master_file)
print("Backup:", backup_file)
print("Dataset shape:", updated_df.shape)
print("Missing descriptions:", updated_df["Description"].isna().sum())


In [ ]:
import pandas as pd
from pathlib import Path

folder = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed"
)

source_file = folder / "retail_sales_description_final.csv"

# Reload the saved dataset so we can inspect the original text
preview_df = pd.read_csv(source_file, low_memory=False)

# Keep the original descriptions for comparison
preview_df["Description_before"] = preview_df["Description"].copy()

# Apply the cleaning function defined in the previous cell
preview_df["Description_after"] = (
    preview_df["Description_before"].apply(clean_product_description)
)

# Find descriptions that would become empty
empty_mask = (
    preview_df["Description_after"].isna()
    | preview_df["Description_after"].str.strip().eq("")
)

print("Descriptions that would become empty:", empty_mask.sum())

print("\nDescriptions to review:")
print(
    preview_df.loc[
        empty_mask,
        ["StockCode", "Description_before", "Description_after"]
    ].to_string(index=False)
)

In [ ]:
import pandas as pd
import re
import shutil
import os
from pathlib import Path
from datetime import datetime

folder = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed"
)

source_file = folder / "retail_sales_description_final.csv"
master_file = folder / "retail_sales_final.csv"

# Load the verified dataset
data = pd.read_csv(source_file, low_memory=False)

original_description = data["Description"].copy()

# Clean the description text
def clean_product_description(description):
    if pd.isna(description):
        return pd.NA

    text = str(description).upper().strip()

    # Remove measurements and numbers
    text = re.sub(
        r'\b\d+(?:\.\d+)?\s*(?:CM|MM|KG|G|GRAMS?|ML|L|INCHES|INCH)\b',
        ' ',
        text
    )
    text = re.sub(r'\d+', ' ', text)

    # Remove punctuation and symbols
    text = re.sub(r'[^A-Z\s]', ' ', text)

    # Remove standalone SET and AND
    text = re.sub(r'\b(?:SET|AND)\b', ' ', text)

    # Normalize spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text if text else pd.NA


cleaned_description = original_description.apply(
    clean_product_description
)

# Find valid descriptions available for each StockCode
candidates = pd.DataFrame({
    "StockCode": data["StockCode"],
    "Description": cleaned_description
}).dropna(subset=["StockCode", "Description"])

# Choose the most frequent cleaned description per StockCode
counts = (
    candidates.groupby(["StockCode", "Description"])
    .size()
    .reset_index(name="frequency")
    .sort_values("frequency", ascending=False)
)

best_description = (
    counts.drop_duplicates("StockCode")
    .set_index("StockCode")["Description"]
)

# Recover descriptions that became empty
missing_mask = cleaned_description.isna()
recovered_description = data["StockCode"].map(best_description)

cleaned_description.loc[missing_mask] = (
    recovered_description.loc[missing_mask]
)

# Identify unresolved descriptions
unresolved = cleaned_description.isna()

# Save unresolved records separately for review
review_file = folder / "retail_descriptions_unresolved_review.csv"

if unresolved.any():
    review_data = data.loc[unresolved].copy()
    review_data["Description_original"] = original_description.loc[
        unresolved
    ]
    review_data.to_csv(review_file, index=False)

# Keep only rows with valid descriptions
final_data = data.loc[~unresolved].copy()
final_data["Description"] = cleaned_description.loc[~unresolved]

# Save to a temporary file and verify before updating the master
temp_file = folder / "retail_sales_final_temp.csv"
final_data.to_csv(temp_file, index=False)

check = pd.read_csv(temp_file, low_memory=False)

if check["Description"].isna().any():
    raise ValueError("Missing descriptions found; master was not updated.")

if check["Description"].str.strip().eq("").any():
    raise ValueError("Empty descriptions found; master was not updated.")

if check["Description"].str.contains(r"\d", na=False).any():
    raise ValueError("Digits found in descriptions; master was not updated.")

# Back up the existing master before replacement
if master_file.exists():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_file = folder / (
        f"retail_sales_final_backup_{timestamp}.csv"
    )
    shutil.copy2(master_file, backup_file)
else:
    backup_file = None

# Update retail_sales_final.csv
os.replace(temp_file, master_file)

print("MASTER DATASET UPDATED SUCCESSFULLY")
print("Rows in source:", len(data))
print("Rows retained:", len(check))
print("Rows removed:", len(data) - len(check))
print("Columns:", len(check.columns))
print("Missing descriptions:", check["Description"].isna().sum())
print("Empty descriptions:", check["Description"].str.strip().eq("").sum())
print("Descriptions containing digits:",
      check["Description"].str.contains(r"\d", na=False).sum())
print("Master file:", master_file)
print("Backup file:", backup_file)
print("Review file:", review_file if unresolved.any() else "Not needed")

print("\nFirst 15 updated descriptions:")
print(check["Description"].head(15).to_string(index=False))

In [ ]:
import pandas as pd
import re
import shutil
from pathlib import Path
from datetime import datetime

# Existing master dataset
file_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_final.csv"
)

# Create a separate backup before changing anything
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = file_path.with_name(
    f"retail_sales_final_before_single_letter_cleanup_{timestamp}.csv"
)

# Load the current master dataset
df = pd.read_csv(file_path, low_memory=False)

rows_before = len(df)

# Remove standalone single-letter words from Description only
df["Description"] = (
    df["Description"]
    .str.replace(r"\b[A-Za-z]\b", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Find descriptions that became empty
empty_mask = (
    df["Description"].isna()
    | df["Description"].eq("")
)

# Preserve the current master file as a backup
shutil.copy2(file_path, backup_path)

# Remove rows only if no description remains after cleaning
rows_removed = int(empty_mask.sum())
df = df.loc[~empty_mask].copy()

# Update the existing master file
df.to_csv(file_path, index=False)

# Verify the saved file
check_df = pd.read_csv(file_path, low_memory=False)

print("Single-letter cleanup completed!")
print("Rows before:", rows_before)
print("Rows removed due to empty descriptions:", rows_removed)
print("Rows remaining:", len(check_df))
print("Columns:", len(check_df.columns))
print("Missing descriptions:", check_df["Description"].isna().sum())
print("Empty descriptions:", check_df["Description"].eq("").sum())

print("\nFirst 20 updated descriptions:")
print(check_df["Description"].head(20).to_string(index=False))

print("\nUpdated master file:", file_path)
print("Backup file:", backup_path)


In [ ]:
import pandas as pd
import re
import shutil
from pathlib import Path
from datetime import datetime

# Existing master dataset
file_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_final.csv"
)

# Load StockCode as text to preserve its letters and format
df = pd.read_csv(file_path, dtype={"StockCode": "string"}, low_memory=False)

if "StockCode" not in df.columns:
    raise ValueError("StockCode column was not found.")

# Create a backup before updating the file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = file_path.with_name(
    f"retail_sales_final_before_stockcode_cleanup_{timestamp}.csv"
)
shutil.copy2(file_path, backup_path)

# Save original values for comparison
original_codes = df["StockCode"].copy()

# Safe cleanup: remove outer spaces and standardize letter case
cleaned_codes = df["StockCode"].str.strip().str.upper()

# Treat blank codes as missing, not as valid codes
cleaned_codes = cleaned_codes.mask(cleaned_codes.eq(""), pd.NA)

# Update StockCode only
df["StockCode"] = cleaned_codes

changed_count = (
    original_codes.fillna("<MISSING>")
    != cleaned_codes.fillna("<MISSING>")
).sum()

# Classify code formats for an audit
def classify_stock_code(value):
    if pd.isna(value):
        return "Missing"

    code = str(value)

    if re.fullmatch(r"\d{5}", code):
        return "Exactly 5 digits"
    elif re.fullmatch(r"\d{5}[A-Z]+", code):
        return "5 digits + letter suffix"
    elif re.fullmatch(r"\d+", code):
        return "Digits, other length"
    elif re.fullmatch(r"[A-Z0-9]+", code):
        return "Other alphanumeric code"
    else:
        return "Contains spaces or punctuation"

format_summary = (
    df["StockCode"]
    .map(classify_stock_code)
    .value_counts(dropna=False)
)

print("StockCode cleanup preview")
print("Rows:", len(df))
print("StockCode values standardized:", changed_count)

print("\nStockCode format summary:")
print(format_summary.to_string())

# Show unusual formats without automatically deleting them
unusual_mask = df["StockCode"].map(
    lambda x: classify_stock_code(x)
).isin(["Contains spaces or punctuation", "Digits, other length"])

print("\nExamples to review:")
print(
    df.loc[unusual_mask, "StockCode"]
    .value_counts()
    .head(30)
    .to_string()
)

# Save the updated master file
df.to_csv(file_path, index=False)

print("\nStockCode cleanup saved!")
print("Updated file:", file_path)
print("Backup file:", backup_path)
print("Missing StockCodes:", df["StockCode"].isna().sum())
print("Unique StockCodes:", df["StockCode"].nunique())

In [ ]:
from pathlib import Path
import pandas as pd

folder = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed"
)

# Compare the master file with its backups
files = sorted(folder.glob("retail_sales_final*.csv"))

print("ROW COUNTS BY FILE")

for file in files:
    row_count = 0

    for chunk in pd.read_csv(
        file,
        usecols=["StockCode"],
        chunksize=200_000,
        dtype={"StockCode": "string"}
    ):
        row_count += len(chunk)

    print(f"{file.name}: {row_count:,} rows")


# Inspect StockCodes outside the two common numeric formats
master_file = folder / "retail_sales_final.csv"

codes = pd.read_csv(
    master_file,
    usecols=["StockCode"],
    dtype={"StockCode": "string"}
)["StockCode"]

standard_format = codes.str.fullmatch(
    r"\d{5}[A-Z]*",
    na=False
)

unusual_codes = codes[~standard_format].value_counts()

print("\nMOST FREQUENT UNUSUAL STOCKCODES")
print(unusual_codes.head(50).to_string())

print("\nUnique unusual codes:", len(unusual_codes))

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed"
)

master_file = folder / "retail_sales_final.csv"

# Read only StockCode and Description
stock_df = pd.read_csv(
    master_file,
    usecols=["StockCode", "Description"],
    dtype={"StockCode": "string"},
    low_memory=False
)

# Identify codes outside the usual numeric format
standard = stock_df["StockCode"].str.fullmatch(
    r"\d{5}[A-Z]*",
    na=False
)

unusual = stock_df.loc[~standard].copy()

# Show each unusual code and its associated descriptions
report = (
    unusual.groupby(["StockCode", "Description"], dropna=False)
    .size()
    .reset_index(name="Rows")
    .sort_values(["StockCode", "Rows"], ascending=[True, False])
)

print("Unusual StockCode and Description combinations:")
print(report.to_string(index=False))

# Save the report separately for review
review_file = folder / "retail_stockcode_review.csv"
report.to_csv(review_file, index=False)

print("\nUnique unusual StockCodes:", unusual["StockCode"].nunique())
print("Review report saved to:", review_file)



In [ ]:
import pandas as pd
import shutil
from pathlib import Path
from datetime import datetime
import os

# Path to your existing master dataset
file_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_final.csv"
)

if not file_path.exists():
    raise FileNotFoundError(f"File not found: {file_path}")

# Load StockCode as text to preserve its original format
df = pd.read_csv(
    file_path,
    dtype={"StockCode": "string"},
    low_memory=False
)

rows_before = len(df)

# Count alphabetic letters in StockCode
letter_count = df["StockCode"].str.count(r"[A-Za-z]")

# Remove rows containing more than one letter
remove_mask = letter_count > 1

removed_df = df.loc[remove_mask].copy()
remaining_df = df.loc[~remove_mask].copy()

# Timestamped backup and removal report
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_path = file_path.with_name(
    f"retail_sales_final_before_multiple_letter_removal_{timestamp}.csv"
)

removed_path = file_path.parent / (
    f"retail_stockcode_rows_removed_{timestamp}.csv"
)

temp_path = file_path.with_name("retail_sales_final_temp.csv")

# Save backup and rows being removed
shutil.copy2(file_path, backup_path)
removed_df.to_csv(removed_path, index=False)
remaining_df.to_csv(temp_path, index=False)

# Verify the temporary dataset before replacing the master file
check_df = pd.read_csv(
    temp_path,
    dtype={"StockCode": "string"},
    low_memory=False
)

if len(check_df) != len(remaining_df):
    raise ValueError("Row count verification failed. Master not updated.")

if (check_df["StockCode"].str.count(r"[A-Za-z]") > 1).any():
    raise ValueError("Some StockCodes still contain multiple letters.")

# Update the master file
os.replace(temp_path, file_path)

print("STOCKCODE FILTERING COMPLETED")
print("Rows before:", rows_before)
print("Rows removed:", len(removed_df))
print("Rows remaining:", len(check_df))
print("Columns:", len(check_df.columns))
print("Unique StockCodes remaining:", check_df["StockCode"].nunique())
print("Backup file:", backup_path)
print("Removed rows report:", removed_path)
print("Updated master file:", file_path)



In [ ]:
import pandas as pd
from pathlib import Path

file_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_final.csv"
)

# Read StockCode as text
df_check = pd.read_csv(
    file_path,
    usecols=["StockCode"],
    dtype={"StockCode": "string"}
)

# Valid means exactly five digits, with no letters or symbols
valid_mask = df_check["StockCode"].str.fullmatch(
    r"\d{5}", na=False
)

invalid_codes = df_check.loc[~valid_mask, "StockCode"]

print("Total rows:", len(df_check))
print("Rows with exactly five digits:", int(valid_mask.sum()))
print("Rows that do not match:", int((~valid_mask).sum()))
print("Missing StockCodes:", int(df_check["StockCode"].isna().sum()))

print("\nUnique invalid StockCodes:")
print(
    invalid_codes.value_counts(dropna=False).head(50).to_string()
)

print("\nTotal unique invalid codes:", invalid_codes.nunique())

In [ ]:
import pandas as pd
import shutil
from pathlib import Path
from datetime import datetime

# Existing master dataset
file_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_final.csv"
)

# Load StockCode as text
df = pd.read_csv(
    file_path,
    dtype={"StockCode": "string"},
    low_memory=False
)

rows_before = len(df)

# Identify codes containing exactly five digits followed by one letter
suffix_mask = df["StockCode"].str.fullmatch(
    r"\d{5}[A-Z]", na=False
)

affected_rows = int(suffix_mask.sum())

# Create a backup before changing the file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = file_path.with_name(
    f"retail_sales_final_before_stockcode_suffix_removal_{timestamp}.csv"
)

shutil.copy2(file_path, backup_path)

# Remove only the final letter from matching StockCodes
df.loc[suffix_mask, "StockCode"] = (
    df.loc[suffix_mask, "StockCode"].str[:-1]
)

# Save the updated master dataset
df.to_csv(file_path, index=False)

# Verify the saved file
check_df = pd.read_csv(
    file_path,
    dtype={"StockCode": "string"},
    low_memory=False
)

print("STOCKCODE SUFFIX CLEANUP COMPLETED")
print("Rows before:", rows_before)
print("Rows after:", len(check_df))
print("StockCodes changed:", affected_rows)
print("Columns:", len(check_df.columns))

print("\nExamples of changed StockCodes:")
print(
    df.loc[suffix_mask, "StockCode"]
    .head(10)
    .to_string(index=False)
)

print("\nUpdated master file:", file_path)
print("Backup file:", backup_path)


In [ ]:
import pandas as pd
from pathlib import Path

file_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_final.csv"
)

df_check = pd.read_csv(
    file_path,
    usecols=["StockCode"],
    dtype={"StockCode": "string"}
)

valid_mask = df_check["StockCode"].str.fullmatch(r"\d{5}", na=False)
invalid = df_check.loc[~valid_mask, "StockCode"]

print("Total rows:", len(df_check))
print("Exactly five digits:", int(valid_mask.sum()))
print("Not exactly five digits:", int((~valid_mask).sum()))

print("\nRemaining unusual StockCodes:")
print(invalid.value_counts().head(30).to_string())

print("\nUnique unusual codes:", invalid.nunique())





In [ ]:
import pandas as pd
import shutil
from pathlib import Path
from datetime import datetime

file_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_final.csv"
)

# Read StockCode as text
df = pd.read_csv(file_path, dtype={"StockCode": "string"})

rows_before = len(df)

# Select rows with exactly five digits
valid = df["StockCode"].str.fullmatch(r"\d{5}", na=False)

# Keep a copy of the rows being removed
removed = df.loc[~valid].copy()
final_df = df.loc[valid].copy()

# Save a backup and a report of removed rows
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_path = file_path.with_name(
    f"retail_sales_final_backup_{timestamp}.csv"
)
removed_path = file_path.parent / (
    f"retail_stockcode_rows_removed_{timestamp}.csv"
)

shutil.copy2(file_path, backup_path)
removed.to_csv(removed_path, index=False)

# Update the master dataset
final_df.to_csv(file_path, index=False)

# Verify the updated file
check = pd.read_csv(file_path, dtype={"StockCode": "string"})

print("Rows before:", rows_before)
print("Rows deleted:", len(removed))
print("Rows remaining:", len(check))
print("All StockCodes are exactly five digits:",
      check["StockCode"].str.fullmatch(r"\d{5}", na=False).all())
print("Updated file:", file_path)
print("Backup:", backup_path)
print("Removed rows report:", removed_path)



In [ ]:

import pandas as pd
import shutil
from pathlib import Path
from datetime import datetime
import os

# Master dataset
file_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_final.csv"
)

# Load the current dataset
df = pd.read_csv(file_path, low_memory=False)

if "InvoiceDate" not in df.columns:
    raise ValueError("InvoiceDate column not found.")

rows_before = len(df)

# Parse dates, interpreting dates such as 01-12-2009 as day-month-year
original_dates = df["InvoiceDate"].copy()
parsed_dates = pd.to_datetime(
    original_dates,
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# Stop if any non-missing date cannot be parsed
invalid_mask = original_dates.notna() & parsed_dates.isna()

if invalid_mask.any():
    print("Some dates could not be parsed. No file was updated.")
    print(original_dates[invalid_mask].head(20).to_string(index=False))
    raise ValueError("Please review the unparsed dates first.")

# Keep date only, in YYYY-MM-DD format
date_only = parsed_dates.dt.strftime("%Y-%m-%d")

# Create a backup before updating the master file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = file_path.with_name(
    f"retail_sales_final_before_date_cleanup_{timestamp}.csv"
)
shutil.copy2(file_path, backup_path)

# Save to a temporary file first
temp_path = file_path.with_name("retail_sales_final_temp.csv")
df["InvoiceDate"] = date_only
df.to_csv(temp_path, index=False)

# Verify that no times remain in InvoiceDate
check_df = pd.read_csv(temp_path, low_memory=False)

if len(check_df) != rows_before:
    raise ValueError("Row count changed. Master file was not updated.")

valid_dates = (
    check_df["InvoiceDate"].isna()
    | check_df["InvoiceDate"].str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)
)

if not valid_dates.all():
    raise ValueError("Date validation failed. Master file was not updated.")

# Replace the existing master file
os.replace(temp_path, file_path)

print("INVOICE DATE CLEANUP COMPLETED")
print("Rows:", len(check_df))
print("Columns:", len(check_df.columns))
print("First 10 dates:")
print(check_df["InvoiceDate"].head(10).to_string(index=False))
print("Updated file:", file_path)
print("Backup file:", backup_path)


In [ ]:
import pandas as pd
import shutil
import os
from pathlib import Path
from datetime import datetime

# Existing master dataset
file_path = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\data\processed\retail_sales_final.csv"
)

# Load the dataset
df = pd.read_csv(file_path, low_memory=False)

if len(df.columns) < 3:
    raise ValueError("The dataset has fewer than three columns.")

# Identify the last three columns
columns_to_remove = df.columns[-3:].tolist()

print("Last three columns:")
print(columns_to_remove)

# Verify every value in each column is False
all_false = (
    df[columns_to_remove]
    .apply(
        lambda col: (
            col.notna().all()
            and col.astype(str).str.strip().str.casefold().eq("false").all()
        )
    )
)

if not all_false.all():
    print("\nSome columns are not entirely False:")
    print(all_false)
    raise ValueError("No changes made. Please review these columns.")

# Create a timestamped backup
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = file_path.with_name(
    f"retail_sales_final_before_last_three_columns_removal_{timestamp}.csv"
)
shutil.copy2(file_path, backup_path)

# Remove only the last three columns
cleaned_df = df.drop(columns=columns_to_remove)

# Save to a temporary file first
temp_path = file_path.with_name("retail_sales_final_temp.csv")
cleaned_df.to_csv(temp_path, index=False)

# Verify the saved result before replacing the master file
check_df = pd.read_csv(temp_path, low_memory=False)

if len(check_df) != len(df):
    raise ValueError("Row count changed. Master file was not updated.")

if len(check_df.columns) != len(df.columns) - 3:
    raise ValueError("Column count verification failed.")

# Update the existing master file
os.replace(temp_path, file_path)

print("\nLAST THREE COLUMNS REMOVED SUCCESSFULLY")
print("Rows preserved:", len(check_df))
print("Columns before:", len(df.columns))
print("Columns after:", len(check_df.columns))
print("Remaining columns:", check_df.columns.tolist())
print("Updated file:", file_path)
print("Backup file:", backup_path)